# Prompt U-Net — GPU Memory Benchmark

Measures peak VRAM on one small and one large 3D volume using `pynvml` (same data as `nvidia-smi`).

**Threshold**: 192³ = 7,077,888 voxels (nnInteractive patch size).
Volumes ≤192³ are small; larger volumes trigger P-UNet tiling overhead.

| Section | Content |
|---------|---------|
| **§1** | Scan NPZ test data — find representative small & large volumes |
| **§2** | GPU memory measurement via pynvml |
| **§3** | Results table |

---
## §1 — Find Representative Volumes

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path().resolve()
project_root = notebook_dir.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
from data.test_data.ds_handler import load_dataset

In [ ]:
# Hardcoded threshold from nnInteractive paper
PATCH_VOXELS = 192 ** 3  # 7,077,888

NPZ_PATHS = [
    project_root / 'data' / 'test_data' / 'FLARE_2022.npz',
    project_root / 'data' / 'test_data' / 'han_seg_ct.npz',
    project_root / 'data' / 'test_data' / 'han_seg_mri.npz',
    project_root / 'data' / 'test_data' / 'HCCTase_ceCT.npz',
    project_root / 'data' / 'test_data' / 'SegRap2023.npz',
    project_root / 'data' / 'test_data' / 'TotalSeg_mri.npz',
]

# Scan all volumes, compute total_voxels for every (pid, axis) combination
candidates = []
for npz_path in NPZ_PATHS:
    ds_name = npz_path.stem
    dataset = load_dataset(str(npz_path))
    for pid, item in dataset.items():
        img = item['image']
        segs = item['segmentations']
        if isinstance(segs, list):
            seg_labels = np.zeros_like(img, dtype=np.int32)
            for li, s in enumerate(segs, 1):
                seg_labels[np.asarray(s) != 0] = li
        else:
            seg_labels = np.asarray(segs).astype(np.int32)
        rois = [r for r in range(1, int(seg_labels.max()) + 1) if (seg_labels == r).sum() > 0]
        if not rois:
            continue
        # For each slicing axis, compute roi_slices and total_voxels
        for axis in range(3):
            sum_axes = tuple(a for a in range(3) if a != axis)
            for roi in rois:
                roi_mask = (seg_labels == roi).astype(np.float32)
                areas = roi_mask.sum(axis=sum_axes)
                roi_slices = int((areas > 0).sum())
                if roi_slices == 0:
                    continue
                h, w = [img.shape[a] for a in range(3) if a != axis]
                total_voxels = roi_slices * h * w
                candidates.append({
                    'npz_path': str(npz_path), 'dataset_name': ds_name,
                    'pid': pid, 'modality': item.get('modality', 'ct'),
                    'axis': axis, 'roi': roi, 'roi_slices': roi_slices,
                    'h': h, 'w': w, 'total_voxels': total_voxels,
                })

print(f'Scanned {len(candidates)} (volume, axis, roi) combinations')

In [ ]:
# Pick one small volume (largest that still fits in 192³) and one large (largest overall)
small = [c for c in candidates if c['total_voxels'] <= PATCH_VOXELS]
large = [c for c in candidates if c['total_voxels'] > PATCH_VOXELS]
print(f'Small candidates: {len(small)}  |  Large candidates: {len(large)}')

# Pick the largest small and the largest large for the stress test
rep_small = max(small, key=lambda c: c['total_voxels'])
rep_large = max(large, key=lambda c: c['total_voxels'])

for label, rep in [('Small', rep_small), ('Large', rep_large)]:
    print(f"\n{label} volume:")
    print(f"  dataset      : {rep['dataset_name']}")
    print(f"  pid          : {rep['pid']}")
    print(f"  modality     : {rep['modality']}")
    print(f"  ROI          : {rep['roi']}")
    print(f"  axis         : {rep['axis']}")
    print(f"  roi_slices   : {rep['roi_slices']}")
    print(f"  in-plane     : {rep['h']}×{rep['w']}")
    print(f"  total_voxels : {rep['total_voxels']:,}")

---
## §2 — GPU Memory Measurement

In [ ]:
import threading
import time

import tensorflow as tf
from pynvml import nvmlInit, nvmlDeviceGetHandleByIndex, nvmlDeviceGetMemoryInfo

from inference.inference_volume import VolumeInference
from inference.ssf import ConfidenceDropStrategy

nvmlInit()
_HANDLE = nvmlDeviceGetHandleByIndex(0)

def _gpu_used_mb():
    return nvmlDeviceGetMemoryInfo(_HANDLE).used / (1024 * 1024)


def measure_peak(inference_fn, poll_interval_s=0.1):
    """Run inference_fn while polling GPU used memory. Returns peak MB."""
    peak = [_gpu_used_mb()]
    stop = threading.Event()
    def poll():
        while not stop.is_set():
            cur = _gpu_used_mb()
            if cur > peak[0]:
                peak[0] = cur
            time.sleep(poll_interval_s)
    t = threading.Thread(target=poll, daemon=True)
    t.start()
    try:
        inference_fn()
    finally:
        stop.set()
        t.join()
    return peak[0]

# Init TF CUDA context and record baseline
_ = tf.zeros([1], dtype=tf.float32)
BASELINE_MB = _gpu_used_mb()
print(f'GPU baseline (idle + CUDA contexts): {BASELINE_MB:.1f} MB')

MODEL_PATH = project_root / 'training' / 'p_unet_332.keras'
print(f'Model: {MODEL_PATH}')

In [ ]:
def load_volume(rep):
    """Load a 3D volume and isolate the selected ROI."""
    dataset = load_dataset(rep['npz_path'])
    item = dataset[rep['pid']]
    img_3d = np.asarray(item['image']).astype(np.float32)
    segs = item['segmentations']
    if isinstance(segs, list):
        seg_labels = np.zeros_like(img_3d, dtype=np.int32)
        for li, s in enumerate(segs, 1):
            seg_labels[np.asarray(s) != 0] = li
    else:
        seg_labels = np.asarray(segs).astype(np.int32)
    seg_3d_binary = (seg_labels == rep['roi']).astype(np.float32)
    # Pick middle slice containing the ROI as prompt
    sum_axes = tuple(a for a in range(3) if a != rep['axis'])
    areas = seg_3d_binary.sum(axis=sum_axes)
    valid = np.where(areas > 0)[0]
    prompt_idx = valid[len(valid) // 2]
    prompt_2d = np.take(seg_3d_binary, prompt_idx, axis=rep['axis'])
    return img_3d, seg_3d_binary, prompt_2d, prompt_idx


def profile_volume(rep):
    """Run full VolumeInference and return peak VRAM above baseline."""
    img_3d, seg_3d_binary, prompt_2d, prompt_idx = load_volume(rep)
    vi = VolumeInference(
        model_path=str(MODEL_PATH),
        modality=rep['modality'],
        normalization='universal',
        ssf_strategy=ConfidenceDropStrategy(drop_fraction=0.05),
        buffer_size=4,
        batch_size=6,
    )
    # Warm-up — absorbs one-shot autotuning / JIT allocations
    _ = vi.run(img_3d=img_3d, seg_3d_binary=seg_3d_binary,
               initial_prompt_2d_seg=prompt_2d,
               prompt_axis=rep['axis'], prompt_idx=prompt_idx)
    # Measurement
    peak = measure_peak(lambda: vi.run(
        img_3d=img_3d, seg_3d_binary=seg_3d_binary,
        initial_prompt_2d_seg=prompt_2d,
        prompt_axis=rep['axis'], prompt_idx=prompt_idx,
    ))
    return peak - BASELINE_MB

In [ ]:
results = {}
for label, rep in [('Small', rep_small), ('Large', rep_large)]:
    print(f'\nProfiling {label} volume: {rep["pid"]} (roi={rep["roi"]}, axis={rep["axis"]})')
    peak_mb = profile_volume(rep)
    results[label] = peak_mb
    print(f'  Peak VRAM (above baseline): {peak_mb:.1f} MB')
    # Clean up
    tf.keras.backend.clear_session()

---
## §3 — Results

In [ ]:
P_UNET_PARAMS = 28.0  # million
P_UNET_DISK   = 108   # MB (.keras)

print(f"{'='*60}")
print(f"  Prompt U-Net GPU Memory")
print(f"  Threshold: {PATCH_VOXELS:,} voxels = 192³")
print(f"  Model: {P_UNET_PARAMS:.0f}M params, {P_UNET_DISK} MB on disk")
print(f"  Batch size: 6")
print(f"{'='*60}")
print(f"  {'Size':<10} {'Volume':<24} {'total_voxels':<16} {'Peak VRAM':<14}")
print(f"  {'-'*64}")
for label, rep in [('Small', rep_small), ('Large', rep_large)]:
    name = f"{rep['dataset_name']}/{rep['pid']}"
    print(f"  {label:<10} {name:<24} {rep['total_voxels']:<16,} {results[label]:.1f} MB")
print(f"{'='*60}")
print(f"\n  Reference (nnInteractive paper, RTX 4090):")
print(f"    Small  (≤192³) : < 6 GB")
print(f"    Large  (>192³) : < 10 GB")
print(f"  This measurement: RTX A6000 (48 GB)")